# 02 — Master Athlete-Session Exploratory Audit (SoccerMon 2020)

This notebook performs a concise descriptive audit of the canonical 2020 athlete-session table produced by Notebook 01.

## Purpose

The goals are to:

1. verify the frozen cohort accounting used by downstream modelling notebooks;
2. describe class imbalance and athlete-level concentration of positive sessions;
3. summarize missingness in contextual variables;
4. inspect selected session-level sensor and contextual distributions descriptively.

This notebook is **not** used for data-dependent feature selection, model tuning, causal inference, or injury-onset localization. Visual comparisons between injury-associated and non-injury athlete-sessions are descriptive only.

The study target remains the same-day athlete-session injury-associated indicator. Because exact within-session injury-onset timestamps are unavailable, no plot in this notebook should be interpreted as evidence of minute-specific injury risk or prospective injury-onset prediction.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 1. Repository paths

Notebook 01 writes the canonical processed tables to `results/processed_data/`. This notebook reads those outputs and writes descriptive artifacts to `results/eda/`.


In [ ]:
def find_project_root(start: Path) -> Path:
    """Resolve the repository root when executed from the root or notebooks directory."""
    start = start.resolve()

    if start.name.lower() == "notebooks":
        return start.parent

    return start


PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_FILE = (
    PROJECT_ROOT
    / "results"
    / "processed_data"
    / "master_session_2020.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "results" / "eda"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_FILE)
print("EDA output directory:", OUTPUT_DIR)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "Canonical master-session input was not found. "
        "Run Notebook 01 successfully before Notebook 02. "
        f"Expected: {INPUT_FILE}"
    )


## 2. Load the canonical athlete-session table

In [ ]:
master_session_2020 = pd.read_csv(
    INPUT_FILE,
    low_memory=False,
)

required_columns = {
    "player_name",
    "team",
    "session_id",
    "injury",
    "total_minutes",
}

missing_required = required_columns - set(master_session_2020.columns)

assert not missing_required, (
    "Canonical master-session table is missing required columns: "
    f"{sorted(missing_required)}"
)

assert not master_session_2020.duplicated(
    ["player_name", "session_id"]
).any()

master_session_2020["injury"] = (
    master_session_2020["injury"]
    .astype(int)
)

print("Master-session table loaded successfully.")
print("Rows:", len(master_session_2020))
print("Columns:", master_session_2020.shape[1])


## 3. Frozen cohort accounting

The following assertions verify that this notebook is operating on the same reconstructed cohort used by the modelling analyses.


In [ ]:
cohort_summary = {
    "athlete_sessions": int(len(master_session_2020)),
    "athletes": int(
        master_session_2020["player_name"].nunique()
    ),
    "injury_associated_sessions": int(
        master_session_2020["injury"].sum()
    ),
    "positive_athletes": int(
        master_session_2020.loc[
            master_session_2020["injury"] == 1,
            "player_name",
        ].nunique()
    ),
}

EXPECTED = {
    "athlete_sessions": 3_743,
    "athletes": 48,
    "injury_associated_sessions": 22,
    "positive_athletes": 5,
}

assert cohort_summary == EXPECTED, (
    "Cohort mismatch. "
    f"Observed: {cohort_summary}; expected: {EXPECTED}"
)

team_summary = (
    master_session_2020
    .groupby("team", as_index=False)
    .agg(
        athlete_sessions=("session_id", "size"),
        athletes=("player_name", "nunique"),
        positive_sessions=("injury", "sum"),
    )
    .sort_values("team")
    .reset_index(drop=True)
)

assert (
    team_summary.loc[
        team_summary["team"] == "TeamA",
        "athlete_sessions",
    ].iloc[0]
    == 2_259
)
assert (
    team_summary.loc[
        team_summary["team"] == "TeamA",
        "positive_sessions",
    ].iloc[0]
    == 22
)
assert (
    team_summary.loc[
        team_summary["team"] == "TeamB",
        "athlete_sessions",
    ].iloc[0]
    == 1_484
)
assert (
    team_summary.loc[
        team_summary["team"] == "TeamB",
        "positive_sessions",
    ].iloc[0]
    == 0
)

print("=== FROZEN COHORT ACCOUNTING ===")
for key, value in cohort_summary.items():
    print(f"{key}: {value:,}")

print("\nTeam-level accounting:")
display(team_summary)


## 4. Class imbalance and positive-athlete concentration

The positive class is extremely sparse. The number of positive athlete-sessions is not equivalent to the number of independent injury events, and repeated positive sessions within an athlete may be statistically and clinically dependent.


In [ ]:
class_summary = (
    master_session_2020["injury"]
    .value_counts(dropna=False)
    .rename_axis("injury")
    .reset_index(name="athlete_sessions")
)

class_summary["fraction"] = (
    class_summary["athlete_sessions"]
    / len(master_session_2020)
)

positive_by_athlete = (
    master_session_2020.loc[
        master_session_2020["injury"] == 1
    ]
    .groupby("player_name", as_index=False)
    .agg(
        positive_sessions=("injury", "sum"),
        team=("team", "first"),
    )
    .sort_values(
        ["positive_sessions", "player_name"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("Outcome distribution:")
display(class_summary)

print("Positive athlete-sessions by athlete:")
display(positive_by_athlete)

print(
    "Positive prevalence:",
    f"{master_session_2020['injury'].mean():.4%}",
)


## 5. Missingness audit

Missingness is summarized for all variables and separately by outcome class. These diagnostics are descriptive and are especially relevant for contextual variables used in PRE-containing sensitivity analyses.


In [ ]:
overall_missingness = (
    master_session_2020
    .isna()
    .mean()
    .rename("overall_missing_fraction")
)

missing_by_outcome = (
    master_session_2020
    .groupby("injury")
    .apply(
        lambda frame: frame.isna().mean(),
        include_groups=False,
    )
    .T
)

missing_by_outcome = missing_by_outcome.rename(
    columns={
        0: "negative_missing_fraction",
        1: "positive_missing_fraction",
    }
)

missingness_table = (
    pd.concat(
        [
            overall_missingness,
            missing_by_outcome,
        ],
        axis=1,
    )
    .sort_values(
        "overall_missing_fraction",
        ascending=False,
    )
)

missingness_table["absolute_outcome_difference"] = (
    missingness_table["positive_missing_fraction"]
    - missingness_table["negative_missing_fraction"]
).abs()

display(
    missingness_table.head(20)
)

missingness_table.to_csv(
    OUTPUT_DIR / "master_session_missingness_2020.csv"
)


## 6. Selected descriptive distributions

To keep the public notebook concise, only a small prespecified set of representative session-level variables is plotted. The purpose is quality control and descriptive inspection, not feature screening.

Boxplots suppress individual outlier markers to improve readability under severe class imbalance. No statistical significance tests are performed.


In [ ]:
SELECTED_SENSOR_VARS = [
    "total_minutes",
    "speed_mean_sess",
    "hr_mean_sess",
    "hacc_mean_sess",
    "inst_acc_mean_sess",
]

SELECTED_CONTEXT_VARS = [
    "srpe_sum",
    "daily_load",
    "weekly_load",
    "atl",
    "ctl28",
    "acwr",
    "sleep_duration",
    "sleep_quality",
]

selected_variables = [
    variable
    for variable in (
        SELECTED_SENSOR_VARS
        + SELECTED_CONTEXT_VARS
    )
    if variable in master_session_2020.columns
]

print(
    "Selected descriptive variables:",
    selected_variables,
)


In [ ]:
def save_injury_boxplot(
    df: pd.DataFrame,
    variable: str,
    output_file: Path,
) -> None:
    """Save a descriptive boxplot by session-level injury-associated status."""

    plot_df = (
        df[["injury", variable]]
        .dropna()
        .copy()
    )

    if plot_df.empty:
        print(f"Skipped {variable}: no non-missing observations.")
        return

    grouped = [
        plot_df.loc[
            plot_df["injury"] == outcome,
            variable,
        ].to_numpy()
        for outcome in [0, 1]
    ]

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.boxplot(
        grouped,
        tick_labels=[
            "Non-injury-associated",
            "Injury-associated",
        ],
        showfliers=False,
    )

    ax.set_title(variable)
    ax.set_ylabel(variable)
    ax.set_xlabel("Athlete-session outcome")
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


for variable in selected_variables:
    save_injury_boxplot(
        master_session_2020,
        variable,
        FIGURE_DIR / f"{variable}_by_injury.png",
    )

print(
    f"Saved {len(selected_variables)} descriptive figures "
    f"to {FIGURE_DIR}"
)


## 7. Descriptive summary by outcome

Median and interquartile-range summaries provide a compact numerical counterpart to the plots. These are descriptive statistics only.


In [ ]:
summary_rows = []

for variable in selected_variables:
    for outcome in [0, 1]:
        values = (
            master_session_2020.loc[
                master_session_2020["injury"] == outcome,
                variable,
            ]
            .dropna()
            .astype(float)
        )

        summary_rows.append(
            {
                "variable": variable,
                "injury": outcome,
                "n_nonmissing": int(len(values)),
                "median": (
                    float(values.median())
                    if len(values)
                    else np.nan
                ),
                "q1": (
                    float(values.quantile(0.25))
                    if len(values)
                    else np.nan
                ),
                "q3": (
                    float(values.quantile(0.75))
                    if len(values)
                    else np.nan
                ),
            }
        )

descriptive_summary = pd.DataFrame(summary_rows)

display(descriptive_summary)

descriptive_summary.to_csv(
    OUTPUT_DIR
    / "selected_variable_descriptive_summary_2020.csv",
    index=False,
)


## 8. Session contribution per athlete

Because athletes contribute unequal numbers of sessions, pooled session-level summaries give greater total influence to athletes with more observations. Downstream modelling therefore complements session-pooled evaluation with athlete-cluster uncertainty and equal-athlete sensitivity analyses.


In [ ]:
sessions_per_athlete = (
    master_session_2020
    .groupby("player_name", as_index=False)
    .agg(
        sessions=("session_id", "size"),
        positive_sessions=("injury", "sum"),
        team=("team", "first"),
    )
    .sort_values(
        ["sessions", "player_name"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(sessions_per_athlete)

print(
    "Sessions per athlete — min / median / max:",
    int(sessions_per_athlete["sessions"].min()),
    float(sessions_per_athlete["sessions"].median()),
    int(sessions_per_athlete["sessions"].max()),
)


## 9. Export descriptive audit tables

In [ ]:
team_summary.to_csv(
    OUTPUT_DIR / "team_summary_2020.csv",
    index=False,
)

class_summary.to_csv(
    OUTPUT_DIR / "class_summary_2020.csv",
    index=False,
)

positive_by_athlete.to_csv(
    OUTPUT_DIR / "positive_sessions_by_athlete_2020.csv",
    index=False,
)

sessions_per_athlete.to_csv(
    OUTPUT_DIR / "sessions_per_athlete_2020.csv",
    index=False,
)

print("Saved descriptive audit artifacts to:", OUTPUT_DIR)


## Output contract and interpretation

A successful run confirms the frozen 2020 cohort and produces:

- `results/eda/team_summary_2020.csv`
- `results/eda/class_summary_2020.csv`
- `results/eda/positive_sessions_by_athlete_2020.csv`
- `results/eda/master_session_missingness_2020.csv`
- `results/eda/selected_variable_descriptive_summary_2020.csv`
- `results/eda/sessions_per_athlete_2020.csv`
- descriptive PNG figures under `results/eda/figures/`

This notebook is intentionally descriptive. It does not define the predictive feature set, select modelling landmarks, estimate causal effects, localize injury onset, or support prospective minute-specific injury-risk claims.
